# Geometry-MMALS G1 v1.1.0
## Functional Geometry-Mediated Routing and Host Allocation

**Status:** complete protocol implementation; not yet executed  
**Depends on:** the replicated v1.0.9 five-seed context-geometry result  
**Primary question:** can the ordered context representation be converted into ordered, causally specific, and functionally meaningful host allocation?

### What the program builds, in concrete terms

Imagine training a system to recognize handwritten digits rotated to $-60^\circ$, $-30^\circ$, $0^\circ$, $30^\circ$, and $60^\circ$. The system sees these conditions progressively, one continual-learning stage at a time.

Inside the system:

- $z_0$ is the **frozen sensory view** of the image;
- $c\in\mathbb{R}^4$ is the **inferred context**, a four-number description of the current situation;
- $r\in\Delta^{H-1}$ is the **route**, a probability distribution assigning work to $H$ hosts;
- $g_h$ are the **hosts**, small specialist transformations;
- $z_M=\sum_h r_h g_h(z_0)$ is the **host synthesis** used for classification.

v1.0.9 established Level 1 evidence: the context space reflects rotation order across five model seeds and generalizes to source identities not used to fit the decoder. It did **not** establish Level 2: the route did not reproducibly inherit that order. v1.1.0 freezes the successful map and tests three mechanisms for reading it.

### Three evidence levels

1. **Representational** — the context reflects the external factor. Achieved as replicated candidate evidence in v1.0.9.
2. **Functional** — ordered context changes cause ordered and meaningful changes in host allocation. This notebook tests that bridge.
3. **Operational** — the bridge improves accuracy, retention, robustness, or cost. Not assumed and not required for the primary v1.1.0 claim.


## 0. Release-integrity setup

The notebook installs the repository in editable mode and verifies the exact scientific version. It does not patch installed source code at runtime. The canonical Python functions live under `src/geometry_mmalls/` and are covered by package tests.


In [ ]:
import os, sys, shutil, importlib, subprocess
from pathlib import Path

EXPECTED_PACKAGE_VERSION = "1.1.0"
EXPECTED_BUILD_REVISION = "functional-routing-bridge-c0-r1"
REPO_URL = "https://github.com/gharbonnier78/geometry-mmalls-g1.git"
REPO_DIR = Path("/content/geometry-mmalls-g1")
SRC_DIR = REPO_DIR / "src"
FORCE_REFRESH = False

if FORCE_REFRESH and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", "--no-deps", "-e", str(REPO_DIR)],
    check=True,
)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
for name in list(sys.modules):
    if name == "geometry_mmalls" or name.startswith("geometry_mmalls."):
        del sys.modules[name]
importlib.invalidate_caches()
import geometry_mmalls

loaded_from = Path(geometry_mmalls.__file__).resolve()
assert loaded_from.parent == (SRC_DIR / "geometry_mmalls").resolve()
assert geometry_mmalls.__version__ == EXPECTED_PACKAGE_VERSION, (
    f"Expected {EXPECTED_PACKAGE_VERSION}, loaded {geometry_mmalls.__version__}. "
    "Push the v1.1.0 package and rerun with FORCE_REFRESH=True."
)
print("Geometry-MMALS", geometry_mmalls.__version__, "from", loaded_from)


## 1. Configuration, profiles, and preregistered treatments

The primary comparison contains no angle-supervised route loss. All routers see the same frozen context, hosts start from the same seed-specific state, and the training budget is matched.

- **R0 MLP:** flexible context-only baseline;
- **R1 Linear:** tests whether a low-capacity map transmits context order;
- **R2 Prototype-energy:** assigns hosts according to distance from learned context prototypes.

The geometric energy of host $h$ is

\[
E_h^{\mathrm{geo}}(c)=\frac{d_C(c,\mu_h)^2}{2\sigma_h^2}+b_h,
\qquad d_C(a,b)=\tfrac12\|a-b\|_2,
\]

and

\[
r_h(c)=\frac{\exp[-E_h^{\mathrm{geo}}(c)/\tau]}
{\sum_k\exp[-E_k^{\mathrm{geo}}(c)/\tau]}.
\]

This is the first controlled component of the broader Energy-Guided Router; cost, memory, stability, host state, goals, and future control are deliberately excluded here.


In [ ]:
from pathlib import Path
import copy, hashlib, json, math, random, time
from dataclasses import asdict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import yaml
from scipy import stats
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import MNIST

from geometry_mmalls.data import FixedAngleMNIST, MultiAngleMNIST
from geometry_mmalls.geometry import (
    paired_route_geometry_loss_stationary,
    paired_context_geometry_loss,
    context_path_spread_loss,
    cross_source_fiber_alignment_loss,
    factor_centroid_geometry_loss,
    normalized_stress,
)
from geometry_mmalls.metrics import (
    bootstrap_mean_ci,
    centroid_geometry_scores,
    grouped_geometry_scores,
    ridge_factor_probe,
)
from geometry_mmalls.model import GeometryMMALS, ResidualHost, SmallConvEncoder
from geometry_mmalls.functional_routing import (
    FunctionalRoutingMMALS,
    LinearContextRouter,
    MLPContextRouter,
    PrototypeEnergyRouter,
    entropic_transport_cost,
    host_functional_diversity_loss,
    host_output_cost_matrix,
    host_specialization,
    host_territory,
    pairwise_functional_route_distances,
    parameter_count,
    reconstruct_route_from_root_probe,
    root_simplex_chord,
    territory_overlap,
    usage_balance_loss,
)

NOTEBOOK_VERSION = "1.1.0"
BUILD_REVISION = "functional-routing-bridge-c0-r1"
base_config = yaml.safe_load(Path("configs/rotated_mnist_g1.yaml").read_text())
protocol = yaml.safe_load(Path("configs/rotated_mnist_g1_v110.yaml").read_text())
assert protocol["build_revision"] == BUILD_REVISION

# development: one seed; pilot: five seeds; qualification: ten seeds.
RUN_PROFILE = os.environ.get("G1_PROFILE", "pilot")
RUN_SECONDARY_MEDIATION_ARM = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SPLIT_SEED = int(protocol["experiment"]["fixed_split_seed"])
SENSORY_SEED = int(protocol["experiment"]["fixed_sensory_seed"])

if RUN_PROFILE == "development":
    MODEL_SEEDS = [0]
    TRAIN_SOURCE_LIMIT, TEST_SOURCE_LIMIT = 256, 320
    SENSORY_SOURCE_LIMIT, SENSORY_TEST_LIMIT = 1500, 750
    SENSORY_EPOCHS, CONTEXT_STAGE_EPOCHS, ROUTER_STAGE_EPOCHS = 1, 1, 1
    SOURCE_BATCH_SIZE, SOURCE_BOOTSTRAP_SAMPLES = 64, 300
    SENSORY_GATE = 0.80
elif RUN_PROFILE == "pilot":
    MODEL_SEEDS = list(map(int, protocol["experiment"]["pilot_model_seeds"]))
    TRAIN_SOURCE_LIMIT, TEST_SOURCE_LIMIT = 512, 640
    SENSORY_SOURCE_LIMIT, SENSORY_TEST_LIMIT = 2000, 1000
    SENSORY_EPOCHS, CONTEXT_STAGE_EPOCHS, ROUTER_STAGE_EPOCHS = 2, 2, 2
    SOURCE_BATCH_SIZE = 64
    SOURCE_BOOTSTRAP_SAMPLES = int(protocol["statistics"]["source_bootstrap_samples"])
    SENSORY_GATE = 0.85
elif RUN_PROFILE == "qualification":
    MODEL_SEEDS = list(map(int, protocol["experiment"]["qualification_model_seeds"]))
    TRAIN_SOURCE_LIMIT = int(base_config["paired_protocol"]["full_source_limit"])
    TEST_SOURCE_LIMIT, SENSORY_SOURCE_LIMIT, SENSORY_TEST_LIMIT = 2500, 12000, 5000
    SENSORY_EPOCHS = int(base_config["training"]["sensory_pretrain_epochs"])
    CONTEXT_STAGE_EPOCHS = ROUTER_STAGE_EPOCHS = int(base_config["training"]["stage_epochs"])
    SOURCE_BATCH_SIZE = int(base_config["paired_protocol"]["source_batch_size"])
    SOURCE_BOOTSTRAP_SAMPLES = int(protocol["statistics"]["source_bootstrap_samples"])
    SENSORY_GATE = 0.95
else:
    raise ValueError("G1_PROFILE must be development, pilot, or qualification")

TRAIN_ANGLES = list(map(float, base_config["data"]["train_angles"]))
CURRICULUM = list(map(float, protocol["experiment"]["curriculum"]))
DENSE_EVAL_ANGLES = list(map(float, protocol["experiment"]["dense_eval_angles"]))
HELDOUT_ANGLES = [angle for angle in DENSE_EVAL_ANGLES if angle not in TRAIN_ANGLES]
CONTEXT_DIM = int(protocol["frozen_representation"]["context_dimension"])
NUM_HOSTS = int(protocol["experiment"]["host_count"])
TAU = float(protocol["training"]["temperature"])
METHOD_SPECS = protocol["routers"]
METHODS = list(METHOD_SPECS)
PRIMARY_CONTRASTS = protocol["primary_contrasts"]

print({
    "version": NOTEBOOK_VERSION,
    "revision": BUILD_REVISION,
    "profile": RUN_PROFILE,
    "device": str(DEVICE),
    "seeds": MODEL_SEEDS,
    "methods": METHODS,
    "secondary_arm": RUN_SECONDARY_MEDIATION_ARM,
})


## 2. Numerical and architectural self-tests

These gates check probability normalization, prototype projection, finite gradients for coincident contexts, and finite entropic transport before any expensive experiment starts.


In [ ]:
torch.manual_seed(110)
probe_context = F.normalize(torch.randn(16, CONTEXT_DIM), dim=-1)
for router in [
    MLPContextRouter(CONTEXT_DIM, NUM_HOSTS),
    LinearContextRouter(CONTEXT_DIM, NUM_HOSTS),
    PrototypeEnergyRouter(CONTEXT_DIM, NUM_HOSTS),
]:
    route = router(probe_context)
    assert route.shape == (16, NUM_HOSTS)
    assert torch.isfinite(route).all()
    assert torch.allclose(route.sum(-1), torch.ones(16), atol=1e-6)

prototype = PrototypeEnergyRouter(CONTEXT_DIM, NUM_HOSTS)
prototype.initialize_from_contexts(probe_context, seed=0)
prototype.project_prototypes_()
assert torch.allclose(prototype.prototypes.norm(dim=-1), torch.ones(NUM_HOSTS), atol=1e-6)

identical = torch.full((8, NUM_HOSTS), 1.0 / NUM_HOSTS, requires_grad=True)
root_loss = root_simplex_chord(identical, identical).mean()
root_loss.backward()
assert identical.grad is not None and torch.isfinite(identical.grad).all()

host_probe = torch.randn(32, NUM_HOSTS, 24)
cost_probe = host_output_cost_matrix(host_probe)
route_a = torch.softmax(torch.randn(32, NUM_HOSTS), dim=-1)
route_b = torch.softmax(torch.randn(32, NUM_HOSTS), dim=-1)
ot_probe = entropic_transport_cost(route_a, route_b, cost_probe, epsilon=0.05)
assert torch.isfinite(ot_probe).all()

print("Router, prototype, root-chord, and Sinkhorn gates: PASS")


## 3. Source-disjoint partitions and frozen sensory grove

Five disjoint test-source partitions are used for distinct measurements:

1. host-function cost construction;
2. mediation-probe fitting;
3. mediation-probe evaluation;
4. tangent fitting;
5. final causal and ablation evaluation.

The sensory encoder is trained once at $0^\circ$ and then frozen for all seeds and treatments.


In [ ]:
root = Path(base_config["data"]["root"])
base_train = MNIST(root=str(root), train=True, download=True)
base_test = MNIST(root=str(root), train=False, download=True)

split_rng = np.random.default_rng(SPLIT_SEED)
train_perm = split_rng.permutation(len(base_train))
test_perm = split_rng.permutation(len(base_test))
sensory_indices = train_perm[:SENSORY_SOURCE_LIMIT].tolist()
train_indices = train_perm[:TRAIN_SOURCE_LIMIT].tolist()
test_indices = test_perm[:TEST_SOURCE_LIMIT].tolist()
sensory_test_indices = test_perm[:SENSORY_TEST_LIMIT].tolist()

partition_arrays = np.array_split(np.asarray(test_indices), 5)
metric_indices = partition_arrays[0].tolist()
probe_fit_indices = partition_arrays[1].tolist()
probe_eval_indices = partition_arrays[2].tolist()
tangent_fit_indices = partition_arrays[3].tolist()
causal_eval_indices = partition_arrays[4].tolist()


def checksum(values):
    return hashlib.sha256(",".join(map(str, values)).encode()).hexdigest()


def generator(seed):
    g = torch.Generator(); g.manual_seed(int(seed)); return g


def fixed_loader(angle, train, indices, shuffle, loader_seed=0, batch_size=128):
    ds = Subset(FixedAngleMNIST(root, angle=angle, train=train, download=True), indices)
    return DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle, num_workers=0,
        generator=generator(loader_seed) if shuffle else None,
    )


def multi_loader(angles, train, indices, shuffle, loader_seed=0, batch_size=None):
    ds = MultiAngleMNIST(root, angles=angles, train=train, indices=indices, download=True)
    return DataLoader(
        ds, batch_size=batch_size or SOURCE_BATCH_SIZE, shuffle=shuffle, num_workers=0,
        generator=generator(loader_seed) if shuffle else None,
    )

split_manifest = {
    "split_seed": SPLIT_SEED,
    "train_sha256": checksum(train_indices),
    "test_sha256": checksum(test_indices),
    "metric_sha256": checksum(metric_indices),
    "probe_fit_sha256": checksum(probe_fit_indices),
    "probe_eval_sha256": checksum(probe_eval_indices),
    "tangent_fit_sha256": checksum(tangent_fit_indices),
    "causal_eval_sha256": checksum(causal_eval_indices),
}

random.seed(SENSORY_SEED); np.random.seed(SENSORY_SEED); torch.manual_seed(SENSORY_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SENSORY_SEED)
latent_dim = int(base_config["model"]["latent_dim"])
encoder = SmallConvEncoder(latent_dim).to(DEVICE)
sensory_head = nn.Linear(latent_dim, 10).to(DEVICE)
sensory_optimizer = torch.optim.AdamW(
    list(encoder.parameters()) + list(sensory_head.parameters()),
    lr=float(base_config["training"]["learning_rate"]),
    weight_decay=float(base_config["training"]["weight_decay"]),
)
sensory_history = []
for epoch in range(SENSORY_EPOCHS):
    total = correct = 0; loss_sum = 0.0
    encoder.train(); sensory_head.train()
    for x, y, _, _ in fixed_loader(0.0, True, sensory_indices, True, SENSORY_SEED * 1000 + epoch):
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = sensory_head(encoder(x)); loss = F.cross_entropy(logits, y)
        sensory_optimizer.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(list(encoder.parameters()) + list(sensory_head.parameters()), 5.0)
        sensory_optimizer.step()
        total += y.numel(); correct += (logits.argmax(1) == y).sum().item()
        loss_sum += float(loss.detach()) * y.numel()
    sensory_history.append({"epoch": epoch, "loss": loss_sum / total, "train_accuracy": correct / total})
    print(sensory_history[-1])

@torch.no_grad()
def evaluate_sensory():
    encoder.eval(); sensory_head.eval(); total = correct = 0
    for x, y, _, _ in fixed_loader(0.0, False, sensory_test_indices, False, batch_size=256):
        x, y = x.to(DEVICE), y.to(DEVICE)
        correct += (sensory_head(encoder(x)).argmax(1) == y).sum().item(); total += y.numel()
    return correct / total

sensory_accuracy = evaluate_sensory()
assert RUN_PROFILE != "qualification" or sensory_accuracy >= SENSORY_GATE
sensory_state = copy.deepcopy(encoder.state_dict())
print("Sensory gate:", sensory_accuracy, "threshold:", SENSORY_GATE)


## 4. Reproduce and freeze the v1.0.9 aligned context checkpoint

For each model seed, this stage reproduces the successful four-dimensional aligned context treatment using the v1.0.9 stationary objective. Only the sensory encoder and context encoder are transferred to v1.1.0. The context encoder is then frozen, so all router treatments receive exactly the same map.

This stage is separated from the routing phase to prevent the context and route from moving together and hiding causality.


In [ ]:
def state_hash(state_dict):
    h = hashlib.sha256()
    for name in sorted(state_dict):
        tensor = state_dict[name].detach().cpu().contiguous()
        h.update(name.encode()); h.update(tensor.numpy().tobytes())
    return h.hexdigest()


def epoch_seed(model_seed, stage, epoch, offset=0):
    return int(model_seed) * 1_000_000 + stage * 10_000 + epoch + int(offset)


def build_context_source_model():
    enc = SmallConvEncoder(latent_dim).to(DEVICE); enc.load_state_dict(sensory_state)
    return GeometryMMALS(
        enc, latent_dim=latent_dim, context_dim=CONTEXT_DIM, num_hosts=NUM_HOSTS,
        host_hidden_dim=int(base_config["model"]["host_hidden_dim"]),
        freeze_encoder=True, bottleneck_hidden_dim=64,
    ).to(DEVICE)


def context_source_forward(model, flat):
    z0 = model.encode(flat)
    raw, context = model.infer_context(z0)
    route = torch.softmax(model.context_bottleneck_router(context) / TAU, dim=-1)
    return model.synthesize(z0, context, route, context_raw=raw)


def train_aligned_context_checkpoint(model_seed):
    torch.manual_seed(model_seed + 1090)
    model = build_context_source_model()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        params,
        lr=float(base_config["training"]["learning_rate"]),
        weight_decay=float(base_config["training"]["weight_decay"]),
    )
    rows = []
    for stage, current_angle in enumerate(CURRICULUM):
        seen = CURRICULUM[:stage + 1]
        totals = {key: 0.0 for key in [
            "loss", "current_ce", "anchor_ce", "route_geo", "context_geo",
            "context_spread", "fiber", "centroid", "host_diversity",
        ]}
        totals.update({"forward_images": 0, "optimizer_steps": 0, "source_examples": 0})
        model.train(); start = time.perf_counter()
        for epoch in range(CONTEXT_STAGE_EPOCHS):
            loader = multi_loader(seen, True, train_indices, True, epoch_seed(model_seed, stage, epoch, 109))
            for views, y, factors, _ in loader:
                batch, angle_count = views.shape[:2]
                flat = views.reshape(-1, *views.shape[2:]).to(DEVICE)
                y, factors = y.to(DEVICE), factors.to(DEVICE)
                trace = context_source_forward(model, flat)
                logits = trace.logits.reshape(batch, angle_count, -1)
                routes = trace.route.reshape(batch, angle_count, -1)
                contexts = trace.context.reshape(batch, angle_count, -1)
                hosts = trace.host_outputs.reshape(batch, angle_count, NUM_HOSTS, -1)
                current_ce = F.cross_entropy(logits[:, -1], y)
                anchor_ce = (
                    F.cross_entropy(
                        logits[:, :-1].reshape(-1, logits.shape[-1]),
                        y[:, None].expand(-1, angle_count - 1).reshape(-1),
                    ) if angle_count > 1 else logits.sum() * 0.0
                )
                route_geo = paired_route_geometry_loss_stationary(routes, factors)
                context_geo = paired_context_geometry_loss(contexts, factors)
                spread = context_path_spread_loss(contexts, 0.05)
                fiber = cross_source_fiber_alignment_loss(contexts, factors)
                centroid = factor_centroid_geometry_loss(contexts, factors)
                diversity = host_functional_diversity_loss(hosts[:, -1])
                loss = (
                    current_ce + 0.10 * anchor_ce + 0.10 * route_geo
                    + 0.10 * context_geo + 0.05 * spread + 0.05 * fiber
                    + 0.05 * centroid + 0.05 * diversity
                )
                components = torch.stack([loss, current_ce, anchor_ce, route_geo, context_geo, spread, fiber, centroid, diversity])
                if not torch.isfinite(components).all():
                    raise FloatingPointError(f"Non-finite context-source loss, seed={model_seed}, stage={stage}")
                optimizer.zero_grad(set_to_none=True); loss.backward()
                grad = torch.nn.utils.clip_grad_norm_(params, 5.0)
                if not torch.isfinite(torch.as_tensor(grad)):
                    raise FloatingPointError("Non-finite context-source gradient")
                optimizer.step()
                totals["source_examples"] += batch; totals["forward_images"] += batch * angle_count
                totals["optimizer_steps"] += 1
                for key, value in {
                    "loss": loss, "current_ce": current_ce, "anchor_ce": anchor_ce,
                    "route_geo": route_geo, "context_geo": context_geo,
                    "context_spread": spread, "fiber": fiber, "centroid": centroid,
                    "host_diversity": diversity,
                }.items():
                    totals[key] += float(value.detach()) * batch
        count = max(totals["source_examples"], 1)
        row = {
            "model_seed": model_seed, "stage": stage, "current_angle": current_angle,
            "seen_angles": json.dumps(seen), "wall_seconds": time.perf_counter() - start,
            "forward_images": totals["forward_images"], "optimizer_steps": totals["optimizer_steps"],
            **{key: totals[key] / count for key in totals if key not in {"forward_images", "optimizer_steps", "source_examples"}},
        }
        rows.append(row); print("context", row)
    checkpoint = {
        "encoder": copy.deepcopy(model.encoder.state_dict()),
        "context_net": copy.deepcopy(model.context_net.state_dict()),
    }
    return model, checkpoint, state_hash(checkpoint["context_net"]), pd.DataFrame(rows)

@torch.no_grad()
def collect_context_bank(source_model, angles, indices):
    contexts = []
    source_model.eval()
    for views, _, _, _ in multi_loader(angles, True, indices, False):
        flat = views.reshape(-1, *views.shape[2:]).to(DEVICE)
        z0 = source_model.encode(flat)
        _, context = source_model.infer_context(z0)
        contexts.append(context.cpu())
    return torch.cat(contexts, dim=0)


## 5. Build matched routing treatments

Hosts, synthesis normalization, and classifier start from one common seed-specific state. Router architectures differ by design, and their parameter counts are reported rather than hidden.

The routing-phase loss is

\[
\mathcal L=\mathcal L_{\mathrm{CE}}+\lambda_A\mathcal L_{\mathrm{anchor}}
+\lambda_B\mathcal L_{\mathrm{balance}}+\lambda_D\mathcal L_{\mathrm{host-div}}.
\]

No angle-supervised route geometry appears in the primary comparison.


In [ ]:
def build_router(method, context_bank, model_seed):
    spec = METHOD_SPECS[method]
    kind = spec["type"]
    torch.manual_seed(model_seed + int(spec["seed_offset"]))
    if kind == "mlp_context_router":
        router = MLPContextRouter(CONTEXT_DIM, NUM_HOSTS, hidden_dim=int(spec["hidden_dim"]))
        init_diag = {}
    elif kind == "linear_context_router":
        router = LinearContextRouter(CONTEXT_DIM, NUM_HOSTS)
        init_diag = {}
    elif kind == "prototype_energy_router":
        router = PrototypeEnergyRouter(
            CONTEXT_DIM, NUM_HOSTS,
            sigma_min=float(spec["sigma_min"]), temperature=TAU,
        )
        init_diag = router.initialize_from_contexts(
            context_bank.to(DEVICE), seed=model_seed + int(spec["seed_offset"]),
            iterations=int(spec["spherical_kmeans_iterations"]),
        )
    else:
        raise ValueError(f"Unknown router type: {kind}")
    return router.to(DEVICE), init_diag


def create_common_host_state(model_seed):
    torch.manual_seed(model_seed + 1100)
    hosts = nn.ModuleList([
        ResidualHost(latent_dim, int(base_config["model"]["host_hidden_dim"]))
        for _ in range(NUM_HOSTS)
    ])
    synthesis_norm = nn.LayerNorm(latent_dim)
    classifier = nn.Linear(latent_dim, 10)
    return {
        "hosts": copy.deepcopy(hosts.state_dict()),
        "synthesis_norm": copy.deepcopy(synthesis_norm.state_dict()),
        "classifier": copy.deepcopy(classifier.state_dict()),
    }


def build_routing_model(method, checkpoint, common_state, context_bank, model_seed):
    enc = SmallConvEncoder(latent_dim); enc.load_state_dict(checkpoint["encoder"])
    context_net = nn.Sequential(nn.Linear(latent_dim, 64), nn.GELU(), nn.Linear(64, CONTEXT_DIM))
    context_net.load_state_dict(checkpoint["context_net"])
    hosts = [ResidualHost(latent_dim, int(base_config["model"]["host_hidden_dim"])) for _ in range(NUM_HOSTS)]
    host_list = nn.ModuleList(hosts); host_list.load_state_dict(common_state["hosts"])
    synthesis_norm = nn.LayerNorm(latent_dim); synthesis_norm.load_state_dict(common_state["synthesis_norm"])
    classifier = nn.Linear(latent_dim, 10); classifier.load_state_dict(common_state["classifier"])
    router, init_diag = build_router(method, context_bank, model_seed)
    model = FunctionalRoutingMMALS(
        enc, context_net, router, hosts, synthesis_norm, classifier,
        freeze_encoder=True, freeze_context=True,
    ).to(DEVICE)
    return model, init_diag


def evaluate_routing_model(model, method, model_seed, stage, angles):
    rows = []; model.eval()
    with torch.no_grad():
        for angle in angles:
            total = correct = 0; nll_sum = 0.0
            for x, y, _, _ in fixed_loader(angle, False, test_indices, False, batch_size=256):
                x, y = x.to(DEVICE), y.to(DEVICE)
                trace = model(x, temperature=TAU)
                total += y.numel(); correct += (trace.logits.argmax(1) == y).sum().item()
                nll_sum += float(F.cross_entropy(trace.logits, y, reduction="sum"))
            rows.append({
                "model_seed": model_seed, "method": method, "stage": stage,
                "angle": float(angle), "angle_type": "train" if angle in TRAIN_ANGLES else "heldout",
                "accuracy": correct / total, "nll": nll_sum / total,
            })
    return rows


def train_router_treatment(model_seed, method, model, checkpoint_hash):
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        params,
        lr=float(base_config["training"]["learning_rate"]),
        weight_decay=float(base_config["training"]["weight_decay"]),
    )
    stage_rows = []; evaluation_rows = []
    for stage, current_angle in enumerate(CURRICULUM):
        seen = CURRICULUM[:stage + 1]
        totals = {key: 0.0 for key in ["loss", "current_ce", "anchor_ce", "balance", "host_diversity"]}
        totals.update({"source_examples": 0, "forward_images": 0, "optimizer_steps": 0})
        start = time.perf_counter(); model.train()
        for epoch in range(ROUTER_STAGE_EPOCHS):
            loader = multi_loader(seen, True, train_indices, True, epoch_seed(model_seed, stage, epoch, 110))
            for views, y, _, _ in loader:
                batch, angle_count = views.shape[:2]
                flat = views.reshape(-1, *views.shape[2:]).to(DEVICE)
                y = y.to(DEVICE)
                trace = model(flat, temperature=TAU)
                logits = trace.logits.reshape(batch, angle_count, -1)
                routes = trace.route.reshape(batch, angle_count, -1)
                hosts = trace.host_outputs.reshape(batch, angle_count, NUM_HOSTS, -1)
                current_ce = F.cross_entropy(logits[:, -1], y)
                anchor_ce = (
                    F.cross_entropy(
                        logits[:, :-1].reshape(-1, logits.shape[-1]),
                        y[:, None].expand(-1, angle_count - 1).reshape(-1),
                    ) if angle_count > 1 else logits.sum() * 0.0
                )
                balance = usage_balance_loss(routes)
                diversity = host_functional_diversity_loss(hosts[:, -1])
                loss = (
                    current_ce
                    + float(protocol["common_loss"]["retention_anchor"]) * anchor_ce
                    + float(protocol["common_loss"]["usage_balance"]) * balance
                    + float(protocol["common_loss"]["host_diversity"]) * diversity
                )
                components = torch.stack([loss, current_ce, anchor_ce, balance, diversity])
                if not torch.isfinite(components).all():
                    raise FloatingPointError(f"Non-finite route loss: seed={model_seed}, method={method}, stage={stage}")
                optimizer.zero_grad(set_to_none=True); loss.backward()
                grad = torch.nn.utils.clip_grad_norm_(params, 5.0)
                if not torch.isfinite(torch.as_tensor(grad)):
                    raise FloatingPointError("Non-finite routing gradient")
                optimizer.step()
                if isinstance(model.router, PrototypeEnergyRouter):
                    model.router.project_prototypes_()
                totals["source_examples"] += batch; totals["forward_images"] += batch * angle_count
                totals["optimizer_steps"] += 1
                for key, value in {"loss": loss, "current_ce": current_ce, "anchor_ce": anchor_ce, "balance": balance, "host_diversity": diversity}.items():
                    totals[key] += float(value.detach()) * batch
        count = max(totals["source_examples"], 1)
        row = {
            "model_seed": model_seed, "method": method, "stage": stage,
            "current_angle": current_angle, "seen_angles": json.dumps(seen),
            "context_checkpoint_hash": checkpoint_hash,
            "router_parameters": parameter_count(model.router),
            "forward_images": totals["forward_images"], "optimizer_steps": totals["optimizer_steps"],
            "wall_seconds": time.perf_counter() - start,
            **{key: totals[key] / count for key in ["loss", "current_ce", "anchor_ce", "balance", "host_diversity"]},
        }
        stage_rows.append(row); print(row)
        evaluation_rows.extend(evaluate_routing_model(model, method, model_seed, stage, DENSE_EVAL_ANGLES))
    return model, pd.DataFrame(stage_rows), pd.DataFrame(evaluation_rows)


## 6. Trace collection and context-preservation audit

A frozen context should be numerically identical across R0, R1, and R2. The notebook records the maximum absolute context difference and the checkpoint hash for every treatment.


In [ ]:
@torch.no_grad()
def collect_trace(model, method, model_seed, angles, indices):
    rows = []; model.eval()
    for views, labels, factors, source_ids in multi_loader(angles, False, indices, False):
        batch, angle_count = views.shape[:2]
        trace = model(views.reshape(-1, *views.shape[2:]).to(DEVICE), temperature=TAU)
        logits = trace.logits.reshape(batch, angle_count, -1)
        arrays = {
            "context": trace.context.reshape(batch, angle_count, -1).cpu().numpy(),
            "route": trace.route.reshape(batch, angle_count, -1).cpu().numpy(),
            "z_mm": trace.z_mm.reshape(batch, angle_count, -1).cpu().numpy(),
            "logits": logits.cpu().numpy(),
        }
        log_probs = F.log_softmax(logits, dim=-1).cpu().numpy()
        for source in range(batch):
            label = int(labels[source])
            for angle_index in range(angle_count):
                rows.append({
                    "model_seed": model_seed, "method": method,
                    "source_index": int(source_ids[source]), "label": label,
                    "angle": float(factors[source, angle_index]),
                    "correct": int(arrays["logits"][source, angle_index].argmax() == label),
                    "true_log_prob": float(log_probs[source, angle_index, label]),
                    "context": arrays["context"][source, angle_index],
                    "route": arrays["route"][source, angle_index],
                    "z_mm": arrays["z_mm"][source, angle_index],
                })
    return pd.DataFrame(rows)


def stack(frame, column):
    return np.stack(frame[column].to_numpy())


def context_preservation_table(trace):
    reference = trace[trace.method == METHODS[0]][["source_index", "angle", "context"]].copy()
    rows = []
    for method in METHODS[1:]:
        other = trace[trace.method == method][["source_index", "angle", "context"]]
        merged = reference.merge(other, on=["source_index", "angle"], suffixes=("_ref", "_other"))
        delta = np.stack(merged.context_ref) - np.stack(merged.context_other)
        rows.append({
            "method": method,
            "reference": METHODS[0],
            "max_abs_context_delta": float(np.max(np.abs(delta))),
            "mean_l2_context_delta": float(np.linalg.norm(delta, axis=1).mean()),
        })
    return pd.DataFrame(rows)


## 7. Nominal and functional route geometry

Historical route geometry treats hosts as fixed simplex coordinates:

\[
d_R(r_a,r_b)=\frac{\|\sqrt{r_a}-\sqrt{r_b}\|_2}{\sqrt2}.
\]

That metric cannot know whether two hosts perform nearly the same function. v1.1.0 therefore constructs a host-function cost matrix

\[
C_{hk}=\mathbb E_x\left[\frac{\|g_h(z_0(x))-g_k(z_0(x))\|_2^2}{d_M}\right]
\]

on a fixed held-out source partition and measures route displacement by entropic optimal transport. The functional metric is evaluation-only in the primary experiment.


In [ ]:
@torch.no_grad()
def build_host_cost(model, angles, indices):
    model.eval(); running = None; count = 0
    for views, _, _, _ in multi_loader(angles, False, indices, False):
        trace = model(views.reshape(-1, *views.shape[2:]).to(DEVICE), temperature=TAU)
        outputs = trace.host_outputs
        delta = outputs[:, :, None, :] - outputs[:, None, :, :]
        batch_cost = torch.square(delta).sum(-1).sum(0) / float(outputs.shape[-1])
        running = batch_cost if running is None else running + batch_cost
        count += outputs.shape[0]
    cost = running / max(count, 1)
    cost = 0.5 * (cost + cost.T); cost.fill_diagonal_(0.0)
    nonzero = cost[cost > 1e-12]
    if nonzero.numel(): cost = cost / nonzero.median()
    return cost.detach().cpu()


def pairwise_factor_distance(factors):
    values = np.asarray(factors, dtype=float)
    return np.abs(values[:, None] - values[None, :])


def pairwise_root_chord_np(routes):
    routes = np.asarray(routes, dtype=float)
    roots = np.sqrt(np.clip(routes, 1e-12, None))
    roots /= np.linalg.norm(roots, axis=1, keepdims=True)
    delta = roots[:, None, :] - roots[None, :, :]
    return np.linalg.norm(delta, axis=-1) / math.sqrt(2.0)


def geometry_score_from_matrices(reference, observed):
    mask = np.triu(np.ones_like(reference, dtype=bool), k=1)
    rho = spearmanr(reference[mask], observed[mask]).statistic
    return float(np.nan_to_num(rho)), normalized_stress(reference, observed, mask=mask)


def grouped_route_geometry(frame, cost=None):
    rows = []
    for source_id, group in frame.groupby("source_index"):
        group = group.sort_values("angle")
        factors = group.angle.to_numpy(float)
        routes = stack(group, "route")
        reference = pairwise_factor_distance(factors)
        if cost is None:
            observed = pairwise_root_chord_np(routes)
            metric = "nominal_root_chord"
        else:
            observed = pairwise_functional_route_distances(
                torch.tensor(routes, dtype=torch.float32),
                cost.float(),
                epsilon=float(protocol["functional_route_metric"]["transport"]["epsilon"]),
                iterations=int(protocol["functional_route_metric"]["transport"]["iterations"]),
            ).numpy()
            metric = "functional_sinkhorn"
        rho, stress = geometry_score_from_matrices(reference, observed)
        rows.append({"source_index": source_id, "metric": metric, "rho": rho, "stress": stress})
    return pd.DataFrame(rows)


def bootstrap_source_mean(values, seed):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(int(seed) % (2 ** 32))
    draws = np.asarray([
        values[rng.integers(0, len(values), len(values))].mean()
        for _ in range(SOURCE_BOOTSTRAP_SAMPLES)
    ])
    return float(values.mean()), float(np.quantile(draws, 0.025)), float(np.quantile(draws, 0.975))


## 8. Low-capacity mediation probes and controls

The probe asks whether a simple map from context to root-route coordinates generalizes to held-out identities and unseen angles. High training reconstruction is not enough.

Controls remove or scramble the factor-relevant context component:

- shuffled root-route targets;
- context with the fitted tangent removed;
- a matched one-dimensional orthogonal component.


In [ ]:
def fit_context_tangent(frame):
    X = stack(frame, "context").astype(float)
    y = frame.angle.to_numpy(float)
    X0 = X - X.mean(0); y0 = y - y.mean()
    ridge = 1e-6 * max(np.trace(X0.T @ X0) / max(X0.shape[1], 1), 1.0)
    tangent = np.linalg.solve(X0.T @ X0 + ridge * np.eye(X0.shape[1]), X0.T @ y0)
    return tangent / max(np.linalg.norm(tangent), 1e-12)


def orthogonal_unit(tangent, seed):
    rng = np.random.default_rng(int(seed) % (2 ** 32))
    while True:
        vector = rng.normal(size=tangent.shape)
        vector -= np.dot(vector, tangent) * tangent
        norm = np.linalg.norm(vector)
        if norm > 1e-12: return vector / norm


def functional_route_error(actual, predicted, cost):
    return float(entropic_transport_cost(
        torch.tensor(actual, dtype=torch.float32),
        torch.tensor(predicted, dtype=torch.float32),
        cost.float(),
        epsilon=float(protocol["functional_route_metric"]["transport"]["epsilon"]),
        iterations=int(protocol["functional_route_metric"]["transport"]["iterations"]),
    ).mean())


def run_mediation_probe(method, model_seed, fit_frame, eval_frame, cost):
    X_train = stack(fit_frame, "context").astype(float)
    routes_train = stack(fit_frame, "route").astype(float)
    y_train = np.sqrt(np.clip(routes_train, 1e-12, None))
    X_eval = stack(eval_frame, "context").astype(float)
    routes_eval = stack(eval_frame, "route").astype(float)
    y_eval = np.sqrt(np.clip(routes_eval, 1e-12, None))
    tangent = fit_context_tangent(fit_frame)
    orth = orthogonal_unit(tangent, model_seed + 811)

    variants = {
        "full_context": (X_train, X_eval, y_train),
        "angle_shuffled": (X_train, X_eval, y_train[np.random.default_rng(model_seed + 812).permutation(len(y_train))]),
        "tangent_removed": (
            X_train - np.outer(X_train @ tangent, tangent),
            X_eval - np.outer(X_eval @ tangent, tangent),
            y_train,
        ),
        "matched_orthogonal_component": (
            np.outer(X_train @ orth, orth),
            np.outer(X_eval @ orth, orth),
            y_train,
        ),
    }
    rows = []
    for variant, (train_x, eval_x, train_y) in variants.items():
        probe = Ridge(alpha=float(protocol["mediation_probe"]["ridge_alpha"])).fit(train_x, train_y)
        prediction_root = probe.predict(eval_x)
        prediction_route = reconstruct_route_from_root_probe(prediction_root)
        rows.append({
            "model_seed": model_seed, "method": method, "variant": variant,
            "root_route_r2": float(r2_score(y_eval, prediction_root, multioutput="variance_weighted")),
            "root_chord_error": float(pairwise_root_chord_np(np.vstack([routes_eval, prediction_route]))[:len(routes_eval), len(routes_eval):].diagonal().mean()),
            "functional_transport_error": functional_route_error(routes_eval, prediction_route, cost),
            "evaluation_samples": len(eval_frame),
        })
    return pd.DataFrame(rows)


## 9. Differential and finite causal mediation

A factor tangent $t$ is fitted on source identities disjoint from causal evaluation. The root-route Jacobian measures local sensitivity:

\[
S_t(c)=\|J_q(c)t(c)\|_2,\qquad
S_\perp(c)=\frac1J\sum_j\|J_q(c)v_j(c)\|_2,
\]

\[
\mathrm{DMR}=\frac{\mathbb E[S_t]}{\mathbb E[S_\perp]+\varepsilon}.
\]

Finite interventions compare tangent and orthogonal movements using both nominal and functional route distances. Prediction identity and true-class log probability are audited so a large route displacement cannot be celebrated if it destroys the task.


In [ ]:
def orthogonal_directions(tangent, count, seed):
    rng = np.random.default_rng(int(seed) % (2 ** 32)); output = []
    while len(output) < count:
        vector = rng.normal(size=tangent.shape)
        vector -= np.dot(vector, tangent) * tangent
        norm = np.linalg.norm(vector)
        if norm > 1e-12: output.append(vector / norm)
    return output


def local_tangent(context, direction):
    projected = direction - (context * direction).sum(-1, keepdim=True) * context
    return F.normalize(projected, dim=-1, eps=1e-8)


def root_route_jvp(router, context, direction):
    context = context.detach().requires_grad_(True)
    direction = direction.detach()
    def function(value):
        return torch.sqrt(router(value, temperature=TAU).clamp_min(1e-12))
    _, jvp = torch.autograd.functional.jvp(function, context, direction, create_graph=False)
    return torch.linalg.vector_norm(jvp, dim=-1)


def source_bootstrap_ratio(frame, numerator, denominator, seed):
    rng = np.random.default_rng(int(seed) % (2 ** 32)); n = len(frame)
    values = []
    for _ in range(SOURCE_BOOTSTRAP_SAMPLES):
        sample = frame.iloc[rng.integers(0, n, n)]
        values.append(sample[numerator].mean() / max(sample[denominator].mean(), 1e-12))
    point = frame[numerator].mean() / max(frame[denominator].mean(), 1e-12)
    return point, float(np.quantile(values, 0.025)), float(np.quantile(values, 0.975))


def causal_mediation_probe(model, method, model_seed, fit_frame, cost):
    tangent_np = fit_context_tangent(fit_frame)
    orthogonal_np = orthogonal_directions(
        tangent_np, int(protocol["causal_protocol"]["orthogonal_controls"]), model_seed + 901
    )
    raw_rows = []
    probe_angle = float(protocol["causal_protocol"]["probe_angle"])
    model.eval()
    for x, labels, _, source_ids in fixed_loader(probe_angle, False, causal_eval_indices, False, batch_size=128):
        x, labels = x.to(DEVICE), labels.to(DEVICE)
        with torch.no_grad():
            base = model(x, temperature=TAU)
            base_logp = F.log_softmax(base.logits, -1).gather(1, labels[:, None]).squeeze(1)
            base_pred = base.logits.argmax(1)
        tangent_global = torch.tensor(tangent_np, dtype=base.context.dtype, device=DEVICE).expand_as(base.context)
        tangent = local_tangent(base.context, tangent_global)
        tangent_sensitivity = root_route_jvp(model.router, base.context, tangent)
        orthogonal_locals = []
        orthogonal_sensitivities = []
        for direction in orthogonal_np:
            global_direction = torch.tensor(direction, dtype=base.context.dtype, device=DEVICE).expand_as(base.context)
            local = local_tangent(base.context, global_direction)
            orthogonal_locals.append(local)
            orthogonal_sensitivities.append(root_route_jvp(model.router, base.context, local))
        orthogonal_sensitivity = torch.stack(orthogonal_sensitivities).mean(0)

        for scale in protocol["causal_protocol"]["intervention_scales"]:
            scale = float(scale)
            with torch.no_grad():
                tangent_context = F.normalize(base.context + scale * tangent, dim=-1)
                tangent_route = model.route_context(tangent_context, temperature=TAU)
                nominal_t = root_simplex_chord(tangent_route, base.route)
                functional_t = entropic_transport_cost(
                    tangent_route, base.route, cost.to(DEVICE),
                    epsilon=float(protocol["functional_route_metric"]["transport"]["epsilon"]),
                    iterations=int(protocol["functional_route_metric"]["transport"]["iterations"]),
                )
                z = torch.einsum("bh,bhd->bd", tangent_route, base.host_outputs)
                new_logits = model.classifier(model.synthesis_norm(z))
                new_logp = F.log_softmax(new_logits, -1).gather(1, labels[:, None]).squeeze(1)
                new_pred = new_logits.argmax(1)

                nominal_controls = []; functional_controls = []
                for local in orthogonal_locals:
                    control_context = F.normalize(base.context + scale * local, dim=-1)
                    control_route = model.route_context(control_context, temperature=TAU)
                    nominal_controls.append(root_simplex_chord(control_route, base.route))
                    functional_controls.append(entropic_transport_cost(
                        control_route, base.route, cost.to(DEVICE),
                        epsilon=float(protocol["functional_route_metric"]["transport"]["epsilon"]),
                        iterations=int(protocol["functional_route_metric"]["transport"]["iterations"]),
                    ))
                nominal_o = torch.stack(nominal_controls).mean(0)
                functional_o = torch.stack(functional_controls).mean(0)

            for index, source_id in enumerate(source_ids):
                raw_rows.append({
                    "model_seed": model_seed, "method": method, "source_index": int(source_id),
                    "scale": scale,
                    "nominal_tangent_change": float(nominal_t[index].cpu()),
                    "nominal_orthogonal_change": float(nominal_o[index].cpu()),
                    "functional_tangent_change": float(functional_t[index].cpu()),
                    "functional_orthogonal_change": float(functional_o[index].cpu()),
                    "tangent_jacobian_sensitivity": float(tangent_sensitivity[index].detach().cpu()),
                    "orthogonal_jacobian_sensitivity": float(orthogonal_sensitivity[index].detach().cpu()),
                    "prediction_preserved": int(base_pred[index] == new_pred[index]),
                    "true_class_log_prob_change": float((new_logp[index] - base_logp[index]).cpu()),
                })
    raw = pd.DataFrame(raw_rows)
    summary = []
    for scale, block in raw.groupby("scale"):
        nominal = source_bootstrap_ratio(block, "nominal_tangent_change", "nominal_orthogonal_change", model_seed + int(abs(scale) * 1000) + 17)
        functional = source_bootstrap_ratio(block, "functional_tangent_change", "functional_orthogonal_change", model_seed + int(abs(scale) * 1000) + 27)
        dmr = source_bootstrap_ratio(block, "tangent_jacobian_sensitivity", "orthogonal_jacobian_sensitivity", model_seed + 37)
        preservation = bootstrap_source_mean(block.prediction_preserved.to_numpy(float), model_seed + 47)
        summary.append({
            "model_seed": model_seed, "method": method, "scale": scale,
            "nominal_csr": nominal[0], "nominal_csr_ci_low": nominal[1], "nominal_csr_ci_high": nominal[2],
            "functional_csr": functional[0], "functional_csr_ci_low": functional[1], "functional_csr_ci_high": functional[2],
            "dmr": dmr[0], "dmr_ci_low": dmr[1], "dmr_ci_high": dmr[2],
            "prediction_preservation": preservation[0],
            "prediction_preservation_ci_low": preservation[1],
            "prediction_preservation_ci_high": preservation[2],
            "mean_true_class_log_prob_change": block.true_class_log_prob_change.mean(),
        })
    return raw, pd.DataFrame(summary)


## 10. Host ecology and ablation

Route-mass differentiation is not enough. A host is functionally specialized only when removing it produces a factor-dependent effect. The notebook therefore exports territory curves, entropy specialization, overlap, functional cost, all-host ablations, top-host removal, and a deterministic random-host control.


In [ ]:
@torch.no_grad()
def host_ecology_tables(model, method, model_seed, dense_trace, cost):
    routes = torch.tensor(stack(dense_trace, "route"), dtype=torch.float32)
    factors = torch.tensor(dense_trace.angle.to_numpy(float), dtype=torch.float32)
    unique, territory = host_territory(routes, factors)
    specialization = host_specialization(territory)
    overlap = territory_overlap(territory)
    territory_rows = []
    for factor_index, angle in enumerate(unique.tolist()):
        for host in range(NUM_HOSTS):
            territory_rows.append({
                "model_seed": model_seed, "method": method, "angle": angle,
                "host": host, "mean_route_mass": float(territory[factor_index, host]),
                "host_specialization": float(specialization[host]),
            })
    overlap_rows = [
        {"model_seed": model_seed, "method": method, "host_a": a, "host_b": b,
         "territory_overlap": float(overlap[a, b]), "functional_cost": float(cost[a, b])}
        for a in range(NUM_HOSTS) for b in range(NUM_HOSTS)
    ]

    ablation_rows = []
    rng = np.random.default_rng(model_seed + 1110)
    for angle in DENSE_EVAL_ANGLES:
        angle_territory = territory[torch.argmin(torch.abs(unique - angle))]
        top_host = int(torch.argmax(angle_territory))
        random_host = int(rng.integers(0, NUM_HOSTS))
        total = 0; baseline_correct = 0
        per_host_correct = np.zeros(NUM_HOSTS, dtype=int)
        for x, y, _, _ in fixed_loader(angle, False, causal_eval_indices, False, batch_size=128):
            x, y = x.to(DEVICE), y.to(DEVICE)
            trace = model(x, temperature=TAU)
            baseline_correct += int((trace.logits.argmax(1) == y).sum()); total += y.numel()
            for host in range(NUM_HOSTS):
                route = trace.route.clone(); route[:, host] = 0.0
                route = route / route.sum(-1, keepdim=True).clamp_min(1e-12)
                z = torch.einsum("bh,bhd->bd", route, trace.host_outputs)
                logits = model.classifier(model.synthesis_norm(z))
                per_host_correct[host] += int((logits.argmax(1) == y).sum())
        baseline_accuracy = baseline_correct / total
        for host in range(NUM_HOSTS):
            ablated_accuracy = per_host_correct[host] / total
            ablation_rows.append({
                "model_seed": model_seed, "method": method, "angle": angle, "host": host,
                "baseline_accuracy": baseline_accuracy, "ablated_accuracy": ablated_accuracy,
                "accuracy_drop": baseline_accuracy - ablated_accuracy,
                "is_top_host": int(host == top_host), "is_random_control": int(host == random_host),
            })
    return pd.DataFrame(territory_rows), pd.DataFrame(overlap_rows), pd.DataFrame(ablation_rows)


## 11. Per-seed execution

Each seed performs two explicit phases:

1. reproduce and freeze one aligned context checkpoint;
2. train R0, R1, and R2 from matched host/classifier states.

The seed then builds method-specific host cost matrices and runs geometry, probes, causal interventions, and ablations.


In [ ]:
def source_geometry_summary(frame, method, model_seed, cost):
    rows = []
    for partition, angles in {"trained": TRAIN_ANGLES, "heldout_dense": HELDOUT_ANGLES}.items():
        sub = frame[frame.angle.isin(angles)]
        nominal = grouped_route_geometry(sub, cost=None)
        functional = grouped_route_geometry(sub, cost=cost)
        for name, table in [("nominal", nominal), ("functional", functional)]:
            for metric in ["rho", "stress"]:
                mean, low, high = bootstrap_source_mean(table[metric], model_seed + (11 if metric == "rho" else 12))
                rows.append({
                    "model_seed": model_seed, "method": method, "partition": partition,
                    "geometry": name, "metric": metric, "mean": mean,
                    "source_ci_low": low, "source_ci_high": high,
                    "source_blocks": len(table),
                })
    return pd.DataFrame(rows)


def paired_source_geometry_effect(source_tables, treatment, control, model_seed):
    rows = []
    for partition in ["trained", "heldout_dense"]:
        for geometry in ["nominal", "functional"]:
            for metric in ["rho", "stress"]:
                t = source_tables[(treatment, partition, geometry)][["source_index", metric]]
                c = source_tables[(control, partition, geometry)][["source_index", metric]]
                merged = t.merge(c, on="source_index", suffixes=("_t", "_c"))
                delta = merged[f"{metric}_t"] - merged[f"{metric}_c"]
                mean, low, high = bootstrap_source_mean(delta, model_seed + 211)
                rows.append({
                    "model_seed": model_seed, "contrast": f"{treatment}_vs_{control}",
                    "partition": partition, "geometry": geometry, "metric": metric,
                    "delta_mean": mean, "delta_ci_low": low, "delta_ci_high": high,
                    "source_blocks": len(merged),
                })
    return rows


def run_seed(model_seed):
    print(f"\n===== MODEL SEED {model_seed} =====")
    random.seed(model_seed); np.random.seed(model_seed); torch.manual_seed(model_seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(model_seed)

    source_model, checkpoint, checkpoint_hash, context_stage = train_aligned_context_checkpoint(model_seed)
    context_bank = collect_context_bank(source_model, TRAIN_ANGLES, train_indices)
    common_state = create_common_host_state(model_seed)
    common_state_hash = state_hash({
        **{f"hosts.{k}": v for k, v in common_state["hosts"].items()},
        **{f"synthesis_norm.{k}": v for k, v in common_state["synthesis_norm"].items()},
        **{f"classifier.{k}": v for k, v in common_state["classifier"].items()},
    })

    models = {}; stage_tables = []; evaluation_tables = []; router_init_rows = []
    for method in METHODS:
        print(f"\n--- {method} ---")
        model, init_diag = build_routing_model(method, checkpoint, common_state, context_bank, model_seed)
        router_init_rows.append({
            "model_seed": model_seed, "method": method,
            "router_parameters": parameter_count(model.router), **init_diag,
        })
        model, stage, evaluation = train_router_treatment(model_seed, method, model, checkpoint_hash)
        models[method] = model; stage_tables.append(stage); evaluation_tables.append(evaluation)

    stage = pd.concat(stage_tables, ignore_index=True)
    evaluation = pd.concat(evaluation_tables, ignore_index=True)
    for stage_id in sorted(stage.stage.unique()):
        block = stage[stage.stage == stage_id]
        assert block.forward_images.nunique() == 1 and block.optimizer_steps.nunique() == 1

    dense_trace = pd.concat([
        collect_trace(models[method], method, model_seed, DENSE_EVAL_ANGLES, test_indices)
        for method in METHODS
    ], ignore_index=True)
    probe_fit_trace = pd.concat([
        collect_trace(models[method], method, model_seed, TRAIN_ANGLES, probe_fit_indices)
        for method in METHODS
    ], ignore_index=True)
    probe_eval_trace = pd.concat([
        collect_trace(models[method], method, model_seed, HELDOUT_ANGLES, probe_eval_indices)
        for method in METHODS
    ], ignore_index=True)
    tangent_fit_trace = pd.concat([
        collect_trace(models[method], method, model_seed, TRAIN_ANGLES, tangent_fit_indices)
        for method in METHODS
    ], ignore_index=True)
    context_preservation = context_preservation_table(dense_trace)

    cost_rows = []; geometry_rows = []; probe_rows = []; causal_raw = []; causal_summary = []
    territory_rows = []; overlap_rows = []; ablation_rows = []; source_geometry_tables = {}
    method_summary_rows = []; forgetting_rows = []
    final_stage = int(evaluation.stage.max())

    for method in METHODS:
        model = models[method]
        method_dense = dense_trace[dense_trace.method == method]
        cost = build_host_cost(model, DENSE_EVAL_ANGLES, metric_indices)
        for a in range(NUM_HOSTS):
            for b in range(NUM_HOSTS):
                cost_rows.append({"model_seed": model_seed, "method": method, "host_a": a, "host_b": b, "cost": float(cost[a, b])})

        # Preserve source-level tables for paired contrasts.
        for partition, angles in {"trained": TRAIN_ANGLES, "heldout_dense": HELDOUT_ANGLES}.items():
            sub = method_dense[method_dense.angle.isin(angles)]
            source_geometry_tables[(method, partition, "nominal")] = grouped_route_geometry(sub, cost=None)
            source_geometry_tables[(method, partition, "functional")] = grouped_route_geometry(sub, cost=cost)
        geometry_rows.append(source_geometry_summary(method_dense, method, model_seed, cost))

        fit = probe_fit_trace[probe_fit_trace.method == method]
        eval_frame = probe_eval_trace[probe_eval_trace.method == method]
        probe_rows.append(run_mediation_probe(method, model_seed, fit, eval_frame, cost))
        causal_fit = tangent_fit_trace[tangent_fit_trace.method == method]
        raw, summary = causal_mediation_probe(model, method, model_seed, causal_fit, cost)
        causal_raw.append(raw); causal_summary.append(summary)
        territory, overlap, ablation = host_ecology_tables(model, method, model_seed, method_dense, cost)
        territory_rows.append(territory); overlap_rows.append(overlap); ablation_rows.append(ablation)

        method_eval = evaluation[evaluation.method == method]
        for angle in TRAIN_ANGLES:
            seen_stage = CURRICULUM.index(angle)
            rows = method_eval[(method_eval.angle == angle) & (method_eval.stage >= seen_stage)]
            best = float(rows.accuracy.max()); final_acc = float(rows[rows.stage == final_stage].accuracy.iloc[0])
            forgetting_rows.append({
                "model_seed": model_seed, "method": method, "angle": angle,
                "best_accuracy": best, "final_accuracy": final_acc, "forgetting": best - final_acc,
            })
        final = method_eval[method_eval.stage == final_stage]
        nominal_train = source_geometry_tables[(method, "trained", "nominal")]
        functional_train = source_geometry_tables[(method, "trained", "functional")]
        method_summary_rows.append({
            "model_seed": model_seed, "method": method,
            "nominal_route_rho": nominal_train.rho.mean(),
            "functional_route_rho": functional_train.rho.mean(),
            "mean_heldout_accuracy": final[final.angle.isin(HELDOUT_ANGLES)].accuracy.mean(),
            "final_trained_accuracy": final[final.angle.isin(TRAIN_ANGLES)].accuracy.mean(),
            "router_parameters": parameter_count(model.router),
        })

    paired_rows = []
    for contrast in PRIMARY_CONTRASTS:
        treatment = contrast["treatment"]; control = contrast["control"]
        paired_rows.extend(paired_source_geometry_effect(source_geometry_tables, treatment, control, model_seed))

    forgetting = pd.DataFrame(forgetting_rows)
    method_summary = pd.DataFrame(method_summary_rows).merge(
        forgetting.groupby(["model_seed", "method"], as_index=False).forgetting.mean().rename(columns={"forgetting": "mean_forgetting"}),
        on=["model_seed", "method"], how="left",
    )

    return {
        "checkpoint_hash": checkpoint_hash,
        "common_state_hash": common_state_hash,
        "context_stage": context_stage,
        "router_initialization": pd.DataFrame(router_init_rows),
        "stage": stage,
        "evaluation": evaluation,
        "trace": dense_trace,
        "context_preservation": context_preservation,
        "host_cost": pd.DataFrame(cost_rows),
        "geometry": pd.concat(geometry_rows, ignore_index=True),
        "paired_geometry": pd.DataFrame(paired_rows),
        "mediation_probe": pd.concat(probe_rows, ignore_index=True),
        "causal_raw": pd.concat(causal_raw, ignore_index=True),
        "causal_summary": pd.concat(causal_summary, ignore_index=True),
        "territory": pd.concat(territory_rows, ignore_index=True),
        "overlap": pd.concat(overlap_rows, ignore_index=True),
        "ablation": pd.concat(ablation_rows, ignore_index=True),
        "forgetting": forgetting,
        "method_summary": method_summary,
    }

all_runs = []
for model_seed in MODEL_SEEDS:
    all_runs.append((model_seed, run_seed(model_seed)))
print("Completed seeds:", [seed for seed, _ in all_runs])


## 12. Seed aggregation and uncertainty

Source identity is the within-seed bootstrap unit. Model seed is the replication unit. Across seeds, the notebook reports paired means, Student-$t$ 95% confidence intervals, standard deviation, and positive-seed fraction.


In [ ]:
def concat_key(key):
    return pd.concat([run[key] for _, run in all_runs], ignore_index=True)

context_stage_df = concat_key("context_stage")
router_initialization_df = concat_key("router_initialization")
stage_df = concat_key("stage")
evaluation_df = concat_key("evaluation")
trace_df = concat_key("trace")
context_preservation_df = concat_key("context_preservation")
host_cost_df = concat_key("host_cost")
geometry_df = concat_key("geometry")
paired_geometry_df = concat_key("paired_geometry")
mediation_probe_df = concat_key("mediation_probe")
causal_raw_df = concat_key("causal_raw")
causal_summary_df = concat_key("causal_summary")
territory_df = concat_key("territory")
overlap_df = concat_key("overlap")
ablation_df = concat_key("ablation")
forgetting_df = concat_key("forgetting")
per_seed_method_df = concat_key("method_summary")

checkpoint_hashes = {str(seed): run["checkpoint_hash"] for seed, run in all_runs}
initial_state_hashes = {str(seed): run["common_state_hash"] for seed, run in all_runs}

compute_summary_df = stage_df.groupby(["model_seed", "method"], as_index=False).agg(
    total_forward_images=("forward_images", "sum"),
    total_optimizer_steps=("optimizer_steps", "sum"),
    total_wall_seconds=("wall_seconds", "sum"),
)


def seed_ci(values, confidence=0.95):
    values = np.asarray(values, dtype=float); n = len(values)
    mean = float(values.mean()); sd = float(values.std(ddof=1)) if n > 1 else 0.0
    if n < 2: return mean, mean, mean, sd, n
    half = float(stats.t.ppf((1 + confidence) / 2, df=n - 1) * sd / math.sqrt(n))
    return mean, mean - half, mean + half, sd, n

aggregate_method_rows = []
for method, block in per_seed_method_df.groupby("method"):
    for metric in ["nominal_route_rho", "functional_route_rho", "mean_heldout_accuracy", "final_trained_accuracy", "mean_forgetting"]:
        mean, low, high, sd, n = seed_ci(block[metric])
        aggregate_method_rows.append({
            "method": method, "metric": metric, "mean": mean,
            "seed_ci_low": low, "seed_ci_high": high, "seed_sd": sd, "seed_count": n,
        })
aggregate_method_df = pd.DataFrame(aggregate_method_rows)

aggregate_effect_rows = []
for keys, block in paired_geometry_df.groupby(["contrast", "partition", "geometry", "metric"]):
    mean, low, high, sd, n = seed_ci(block.delta_mean)
    favorable = block.delta_mean > 0 if keys[3] == "rho" else block.delta_mean < 0
    aggregate_effect_rows.append({
        "contrast": keys[0], "partition": keys[1], "geometry": keys[2], "metric": keys[3],
        "mean_seed_effect": mean, "seed_ci_low": low, "seed_ci_high": high,
        "seed_sd": sd, "seed_count": n,
        "favorable_seed_fraction": float(np.mean(favorable)),
    })
aggregate_effect_df = pd.DataFrame(aggregate_effect_rows)

probe_seed_df = mediation_probe_df.pivot_table(
    index=["model_seed", "method"], columns="variant", values=["root_route_r2", "root_chord_error", "functional_transport_error"]
).reset_index()
probe_seed_df.columns = ["_".join([str(part) for part in col if str(part)]) if isinstance(col, tuple) else col for col in probe_seed_df.columns]

causal_seed_df = (
    causal_summary_df[causal_summary_df.scale != 0]
    .groupby(["model_seed", "method"], as_index=False)
    .agg(
        mean_nominal_csr=("nominal_csr", "mean"),
        mean_functional_csr=("functional_csr", "mean"),
        min_functional_csr_ci_low=("functional_csr_ci_low", "min"),
        mean_dmr=("dmr", "mean"),
        min_prediction_preservation_ci_low=("prediction_preservation_ci_low", "min"),
    )
)

ablation_seed_df = ablation_df.groupby(["model_seed", "method"], as_index=False).agg(
    mean_top_host_drop=("accuracy_drop", lambda values: float(ablation_df.loc[values.index].query("is_top_host == 1").accuracy_drop.mean())),
    mean_random_host_drop=("accuracy_drop", lambda values: float(ablation_df.loc[values.index].query("is_random_control == 1").accuracy_drop.mean())),
)

aggregate_method_df.head(), aggregate_effect_df.head(), causal_seed_df.head()


## 13. Candidate gates and falsification logic

A positive result must pass mechanism-specific gates without hiding operational degradation.

- If R2 improves neither nominal nor functional route mediation, prototype routing fails to build the bridge.
- If functional geometry improves while nominal geometry does not, the old simplex metric was incomplete.
- If route mediation improves but host ablations remain factor-independent, the bridge stops at routing.
- If host effects improve while accuracy and retention remain neutral, v1.1.0 reaches functional but not operational geometry.
- If accuracy improves without mediation evidence, the system may be useful, but the geometric mechanism is not validated.


In [ ]:
def get_effect(contrast, partition, geometry, metric):
    rows = aggregate_effect_df[
        (aggregate_effect_df.contrast == contrast)
        & (aggregate_effect_df.partition == partition)
        & (aggregate_effect_df.geometry == geometry)
        & (aggregate_effect_df.metric == metric)
    ]
    if rows.empty:
        raise KeyError((contrast, partition, geometry, metric))
    return rows.iloc[0]

primary_name = "r2_prototype_energy_vs_r0_mlp"
nominal_effect = get_effect(primary_name, "trained", "nominal", "rho")
functional_effect = get_effect(primary_name, "trained", "functional", "rho")
r2_causal = causal_seed_df[causal_seed_df.method == "r2_prototype_energy"]
r2_probe = mediation_probe_df[mediation_probe_df.method == "r2_prototype_energy"]
full_probe = r2_probe[r2_probe.variant == "full_context"]
controls = r2_probe[r2_probe.variant != "full_context"]
probe_pass_by_seed = []
for seed in MODEL_SEEDS:
    full = full_probe[full_probe.model_seed == seed].iloc[0]
    control = controls[controls.model_seed == seed]
    probe_pass_by_seed.append(
        full.root_route_r2 > control.root_route_r2.max()
        and full.functional_transport_error < control.functional_transport_error.min()
    )

r2_ablation = ablation_df[ablation_df.method == "r2_prototype_energy"]
mean_top_drop = r2_ablation[r2_ablation.is_top_host == 1].groupby("model_seed").accuracy_drop.mean()
mean_random_drop = r2_ablation[r2_ablation.is_random_control == 1].groupby("model_seed").accuracy_drop.mean()
host_ecology_positive = float(np.mean(mean_top_drop > mean_random_drop))

r2_perf = per_seed_method_df[per_seed_method_df.method == "r2_prototype_energy"].set_index("model_seed")
r0_perf = per_seed_method_df[per_seed_method_df.method == "r0_mlp"].set_index("model_seed")
accuracy_delta = r2_perf.mean_heldout_accuracy - r0_perf.mean_heldout_accuracy
forgetting_delta = r2_perf.mean_forgetting - r0_perf.mean_forgetting
equivalence_margin = float(protocol["candidate_gates"]["operational_equivalence_margin"])

candidate_gates = pd.DataFrame([
    {
        "gate": "C0_context_frozen_and_preserved",
        "passed": bool(context_preservation_df.max_abs_context_delta.max() <= 1e-6),
        "status": "protocol_integrity",
    },
    {
        "gate": "C0_equal_compute",
        "passed": bool(compute_summary_df.groupby("model_seed").apply(lambda b: b.total_forward_images.nunique() == 1 and b.total_optimizer_steps.nunique() == 1).all()),
        "status": "protocol_integrity",
    },
    {
        "gate": "C1_nominal_route_mediation",
        "passed": bool(nominal_effect.seed_ci_low > 0 and nominal_effect.favorable_seed_fraction >= 0.80),
        "status": "candidate_only",
    },
    {
        "gate": "C1_functional_route_mediation",
        "passed": bool(functional_effect.seed_ci_low > 0 and functional_effect.favorable_seed_fraction >= 0.80),
        "status": "candidate_only",
    },
    {
        "gate": "C1_probe_generalization",
        "passed": bool(np.mean(probe_pass_by_seed) >= 0.80),
        "status": "candidate_only",
    },
    {
        "gate": "C4_functional_causal_specificity",
        "passed": bool((r2_causal.min_functional_csr_ci_low > 1.0).mean() >= 0.80 and r2_causal.mean_functional_csr.mean() > 1.5),
        "status": "candidate_only",
    },
    {
        "gate": "C3_host_ecology",
        "passed": bool(host_ecology_positive >= 0.80),
        "status": "candidate_only",
    },
    {
        "gate": "C4_identity_preservation",
        "passed": bool((r2_causal.min_prediction_preservation_ci_low >= 0.95).mean() >= 0.80),
        "status": "candidate_only",
    },
    {
        "gate": "C6_operational_non_degradation",
        "passed": bool((np.abs(accuracy_delta) <= equivalence_margin).mean() >= 0.80 and (forgetting_delta <= equivalence_margin).mean() >= 0.80),
        "status": "equivalence_not_superiority",
    },
    {"gate": "C5_memory_transport", "passed": False, "status": "not_implemented"},
    {"gate": "operational_superiority", "passed": False, "status": "not_preregistered_as_primary"},
])
candidate_gates


## 14. Optional preregistered secondary mediation arm

This arm is disabled by default and must never be pooled with the architecture-only result. It asks a different question: can mediation be trained directly after the primary R2 result is reported?

A warm-up phase builds a frozen host-function cost matrix. The secondary margin loss is

\[
\mathcal L_{\mathrm{med}}=
\operatorname{ReLU}\left(
 m_{\mathrm{med}}-d_{\mathrm{func}}(r(c+\epsilon t),r(c))
 +d_{\mathrm{func}}(r(c+\epsilon v_\perp),r(c))
\right).
\]

The full training implementation is packaged as a callable helper, but execution requires `RUN_SECONDARY_MEDIATION_ARM=True` and exports to a separate method namespace.


In [ ]:
def functional_mediation_margin_loss(router, contexts, tangent, orthogonal, cost, scale=0.25, margin=0.05):
    tangent_local = local_tangent(contexts, tangent.expand_as(contexts))
    orthogonal_local = local_tangent(contexts, orthogonal.expand_as(contexts))
    base_route = router(contexts, temperature=TAU)
    tangent_route = router(F.normalize(contexts + scale * tangent_local, dim=-1), temperature=TAU)
    orthogonal_route = router(F.normalize(contexts + scale * orthogonal_local, dim=-1), temperature=TAU)
    tangent_change = entropic_transport_cost(
        tangent_route, base_route, cost,
        epsilon=float(protocol["functional_route_metric"]["transport"]["epsilon"]),
        iterations=int(protocol["functional_route_metric"]["transport"]["iterations"]),
    )
    orthogonal_change = entropic_transport_cost(
        orthogonal_route, base_route, cost,
        epsilon=float(protocol["functional_route_metric"]["transport"]["epsilon"]),
        iterations=int(protocol["functional_route_metric"]["transport"]["iterations"]),
    )
    return F.relu(float(margin) - tangent_change + orthogonal_change).mean()

if RUN_SECONDARY_MEDIATION_ARM:
    print("Secondary arm requested. Run it only after archiving the primary report; use a separate method and output directory.")
else:
    print("Secondary mediation arm: preregistered but disabled for the primary run.")


## 15. Export immutable evidence

Every seed exports checkpoint hashes, initial-state hashes, compute, nominal and functional geometry, host costs, probes, causal results, host ecology, accuracy, NLL, forgetting, and claim status. The top-level manifest explicitly states that v1.1.0 is not yet a final C1–C6 qualification.


In [ ]:
try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
except Exception:
    git_sha = "unknown"

root_out = Path("results/v1_1_0")
root_out.mkdir(parents=True, exist_ok=True)
for model_seed, run in all_runs:
    seed_out = root_out / f"seed_{model_seed}"
    seed_out.mkdir(parents=True, exist_ok=True)
    for filename, key in {
        "context_pretraining_stage_logs.csv": "context_stage",
        "router_initialization.csv": "router_initialization",
        "routing_stage_logs.csv": "stage",
        "staged_evaluation.csv": "evaluation",
        "dense_trace.pkl": "trace",
        "context_preservation.csv": "context_preservation",
        "host_cost_matrices.csv": "host_cost",
        "route_geometry.csv": "geometry",
        "paired_route_geometry_effects.csv": "paired_geometry",
        "mediation_probe_controls.csv": "mediation_probe",
        "causal_mediation_raw.csv": "causal_raw",
        "causal_mediation_summary.csv": "causal_summary",
        "host_territory.csv": "territory",
        "host_overlap_and_cost.csv": "overlap",
        "host_ablation.csv": "ablation",
        "forgetting.csv": "forgetting",
        "method_summary.csv": "method_summary",
    }.items():
        value = run[key]
        if filename.endswith(".pkl"):
            value.to_pickle(seed_out / filename)
        else:
            value.to_csv(seed_out / filename, index=False)
    (seed_out / "checkpoint_hashes.json").write_text(json.dumps({
        "context_checkpoint_hash": run["checkpoint_hash"],
        "common_host_classifier_initial_state_hash": run["common_state_hash"],
    }, indent=2), encoding="utf-8")

for filename, frame in {
    "context_pretraining_stage_logs.csv": context_stage_df,
    "router_initialization.csv": router_initialization_df,
    "routing_stage_logs.csv": stage_df,
    "staged_evaluation.csv": evaluation_df,
    "context_preservation.csv": context_preservation_df,
    "host_cost_matrices.csv": host_cost_df,
    "route_geometry.csv": geometry_df,
    "paired_route_geometry_effects.csv": paired_geometry_df,
    "aggregate_seed_effects.csv": aggregate_effect_df,
    "mediation_probe_controls.csv": mediation_probe_df,
    "causal_mediation_summary.csv": causal_summary_df,
    "causal_seed_summary.csv": causal_seed_df,
    "host_territory.csv": territory_df,
    "host_overlap_and_cost.csv": overlap_df,
    "host_ablation.csv": ablation_df,
    "forgetting.csv": forgetting_df,
    "per_seed_method_summary.csv": per_seed_method_df,
    "aggregate_method_summary.csv": aggregate_method_df,
    "compute_summary.csv": compute_summary_df,
    "gate_summary.csv": candidate_gates,
    "sensory_history.csv": pd.DataFrame(sensory_history),
}.items():
    frame.to_csv(root_out / filename, index=False)

claim_manifest = {
    "version": NOTEBOOK_VERSION,
    "status": "C0 complete implementation; empirical v1.1.0 claims pending execution",
    "eligible_if_gates_pass": [
        "functional context-to-route mediation",
        "prototype-energy structural routing candidate",
        "host ecology candidate",
    ],
    "non_claims": protocol["non_claims"],
}
(root_out / "claim_manifest.json").write_text(json.dumps(claim_manifest, indent=2), encoding="utf-8")

manifest = {
    "notebook_version": NOTEBOOK_VERSION,
    "build_revision": BUILD_REVISION,
    "package_version": geometry_mmalls.__version__,
    "git_sha": git_sha,
    "run_profile": RUN_PROFILE,
    "model_seeds": MODEL_SEEDS,
    "context_checkpoint_hashes": checkpoint_hashes,
    "common_initial_state_hashes": initial_state_hashes,
    "split_manifest": split_manifest,
    "sensory_accuracy": sensory_accuracy,
    "methods": METHOD_SPECS,
    "primary_contrasts": PRIMARY_CONTRASTS,
    "functional_metric_evaluation_only": True,
    "secondary_mediation_arm_executed": RUN_SECONDARY_MEDIATION_ARM,
    "source_v109_results": "results/v1_0_9",
    "article": "docs/reports/Geometry_MMALS_G1_v1_0_9_Results_and_v1_1_0_Specification.pdf",
    "reviewer_report": "docs/reports/Geometry_MMALS_G1_Status_and_Perspective_Reviewer_Report_v1_2.pdf",
    "warning": "No final C1-C6 qualification may be claimed from C0 implementation alone.",
    "timestamp": time.time(),
}
(root_out / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

import platform
(root_out / "environment.txt").write_text(
    "\n".join([
        f"python={sys.version}", f"platform={platform.platform()}",
        f"torch={torch.__version__}", f"device={DEVICE}",
    ]), encoding="utf-8"
)
print("Exported v1.1.0 evidence to", root_out.resolve())


## 16. Interpretation discipline and what remains after v1.1.0

### A successful v1.1.0 result would mean

The ordered context is not merely a readable map: its geometry influences how work is distributed among hosts, and that influence survives source-disjoint probes and causal interventions.

### It would still not prove

- operational superiority;
- mature host specialization;
- transport of memory through continual updates;
- bottom-up host feedback;
- full multi-objective energy control;
- RL or forward–backward control;
- distributed mycelial routing;
- quantum or quantum-inspired advantage.

### Roadmap

- **G1.2 Host ecology:** heterogeneous roles, overlap, redundancy, ablation, recovery.
- **G1.3 Memory transport:** preserve or deliberately move geometric and functional relations through continual updates.
- **G2 Full Energy-Guided Router:** add host uncertainty, cost, memory risk, stability, and goal conditioning to $E_h^{\mathrm{geo}}$.
- **G2.1 Bottom-up feedback:** hosts report confidence, novelty, saturation, capacity, and error.
- **G2.2 RL / predictive control:** optimize future consequences and reachable goals.
- **G3 Phase-aware / quantum-inspired routing:** amplitudes, phases, interference, and density matrices only after functional geometry is established.

The immediate scientific path is therefore understandable and falsifiable:

\[
\text{external factor}\rightarrow\text{context geometry}
\rightarrow\text{route mediation}\rightarrow\text{host function}
\rightarrow\text{operational value}.
\]


## 17. Qualification checklist

Before publication-level claims:

- [ ] run the five-seed pilot without manual source patching;
- [ ] verify exact context preservation and matched compute;
- [ ] archive every host cost matrix and partition hash;
- [ ] report R2–R0 and R2–R1, not only absolute R2 scores;
- [ ] keep nominal and functional route geometry separate;
- [ ] keep architecture-only and mediation-trained arms separate;
- [ ] report negative results and operational equivalence intervals;
- [ ] run ten seeds only after the functional mediation gate passes;
- [ ] update the reviewer report only from immutable exported CSVs.
